In [1]:
!pip install datasets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 37.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 70.1 MB/s  0:00:006m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 47.8 MB/s  0:00:00m0:00:01
  Attempting uninstall: fsspec0m━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━  2/14 [propcache]
    Found existing installation: fsspec 2025.12.0━━━━━━━━━━━━━  2/14 [propcache]
    Uninstalling fsspec-2025.12.0:━━━━━━━━━━━━━━━━━━━━━━━━━━━━  2/14 [propcache]
      Successfully uninstalled fsspec-2025.12.0━━━━━━━━━━━━━━━  2/14 [propcache]
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14/14 [datasets]/14 [datasets]]ss]

[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python3 -m pip install --upgrade pip


In [1]:
from datasets import load_dataset

dataset = load_dataset("wikimedia/wikipedia", "20231101.tr",streaming=True)


/usr/local/python/3.12.1/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
from datasets import load_dataset
import re



TARGET_WORD_COUNT = 5_000_000   # Kaç kelime istiyorsun?
MIN_ARTICLE_WORDS = 200         # Çok kısa sayfaları alma
OUTPUT_FILE = "wikipedia_tr_5m_words.txt"

def clean_text(text):
    text = re.sub(r"\s+", " ", text)  # fazla boşlukları temizle
    text = text.strip()
    return text



dataset = load_dataset("wikimedia/wikipedia", "20231101.tr",streaming=True)

total_words = 0

with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
    for example in dataset["train"]:
        text = clean_text(example["text"])
        word_count = len(text.split())

        if word_count < MIN_ARTICLE_WORDS:
            continue

        if total_words >= TARGET_WORD_COUNT:
            break

        f.write(text + "\n")
        total_words += word_count

        if total_words % 500_000 < word_count:
            print(f"Toplam kelime: {total_words}")

print(f"\nBitti  Toplam kelime: {total_words}")
print(f"Dosya kaydedildi: {OUTPUT_FILE}")


Toplam kelime: 500388
Toplam kelime: 1001574
Toplam kelime: 1500949
Toplam kelime: 2000259
Toplam kelime: 2500425
Toplam kelime: 3000227
Toplam kelime: 3500417
Toplam kelime: 4000409
Toplam kelime: 4503239
Toplam kelime: 5000659

Bitti  Toplam kelime: 5000659
Dosya kaydedildi: wikipedia_tr_5m_words.txt


In [1]:
!pip install datasets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 38.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 47.8 MB/s  0:00:00m0:00:01:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 88.8 MB/s  0:00:00
  Attempting uninstall: fsspec━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━  3/14 [multidict]
    Found existing installation: fsspec 2026.2.0━━━━━━━━━━━━━━  3/14 [multidict]
    Uninstalling fsspec-2026.2.0:━━━━━━━━━━━━━━━━━━━━━━━━━━━━━  3/14 [multidict]
      Successfully uninstalled fsspec-2026.2.0━━━━━━━━━━━━━━━━  3/14 [multidict]
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14/14 [datasets]/14 [datasets]ess]

[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python3 -m pip install --upgrade pip


In [ ]:
import re
from datasets import load_dataset
import unicodedata


TARGET_WORD_COUNT = 100_000_000  
MIN_ARTICLE_WORDS = 200
OUTPUT_FILE = "wikipedia_tr_100m_words.txt"

LOG_EVERY = 5_000_000 # her 5 milyon tokende log bas

TOKEN_PER_WORD_EST= 1.3 # tahmin edilen token/kelime oranı



def clean_text(text):

    # Unicode normalize
    text = unicodedata.normalize("NFKC", text)

    # HTML entity temizle
    text = re.sub(r"&[a-z]+;", " ", text)

    # URL temizle
    text = re.sub(r"http\S+|www\S+", " ", text)

    # Referans [1], [23]
    text = re.sub(r"\[\d+\]", " ", text)

    # Çoklu eşittir başlıkları ===== Başlık =====
    text = re.sub(r"=+", " ", text)

    # File:, Image: satırlarını sil
    text = re.sub(r"(File|Image):\S+", " ", text)

    # Fazla noktalama tekrarlarını azalt
    text = re.sub(r"([!?.,])\1+", r"\1", text)

    # Tek harfli satırları temizle
    text = re.sub(r"\n.\n", "\n", text)

    # Fazla boşluk
    text = re.sub(r"\s+", " ", text)

    text = text.strip()

    return text
def estimate_tokens(word_count):
    return int(word_count * TOKEN_PER_WORD_EST)

dataset = load_dataset("wikimedia/wikipedia", "20231101.tr",streaming=True)

total_words = 0
total_tokens = 0
article_count = 0

print("streaming başlıyor...")

with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
    for example in dataset["train"]:
        text= clean_text(example["text"])

        word_count = len(text.split())

        if word_count < MIN_ARTICLE_WORDS:
            continue
        token_est= estimate_tokens(word_count)

        if total_tokens > TARGET_WORD_COUNT:
            break
        f.write(text + "\n")

        total_words += word_count
        total_tokens += token_est
        article_count += 1

        if total_tokens %  LOG_EVERY <token_est:
            print(f"Articles: {article_count}  Total Words: {total_words}  Estimated Tokens: {total_tokens}")


    print("\nTamamlandı!")
    print(f"Toplam makale: {article_count}")
    print(f"Toplam kelime: {total_words:,}")
    print(f"Tahmini token: {total_tokens:,}")
    print(f"Dosya: {OUTPUT_FILE}")



In [ ]:
import re
import unicodedata
from datasets import load_dataset

TARGET_TOKENS = 600_000_000
TOKEN_EST = 1.3
MIN_WORDS = 50
OUTPUT_FILE = "turkish_llm_dataset.txt"
LOG_EVERY_DOCS = 50_000

def clean_text(text):
    text = unicodedata.normalize("NFKC", str(text))
    text = re.sub(r"http\S+", " ", text)
    text = re.sub(r"\[\d+\]", " ", text)
    text = re.sub(r"\{\{.*?\}\}", " ", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()

def estimate_tokens(words):
    return int(words * TOKEN_EST)

def process(dataset, field, file, stats):
    for sample in dataset:
        try:
            text = clean_text(sample.get(field, ""))
            if not text:
                continue

            words = len(text.split())
            if words < MIN_WORDS:
                continue

            tokens = estimate_tokens(words)
            if stats["tokens"] >= TARGET_TOKENS:
                return True

            file.write(text + "\n")
            stats["tokens"] += tokens
            stats["docs"] += 1

            if stats["docs"] % LOG_EVERY_DOCS == 0:
                print(f"docs: {stats['docs']:,} | tokens(est): {stats['tokens']:,}")
        except Exception as row_error:
            # Tekil bozuk örnekler pipeline'ı durdurmasın
            print(f"Satır atlandı: {type(row_error).__name__}: {row_error}")
            continue
    return False

def try_load_stream(source):
    try:
        ds = load_dataset(
            source["path"],
            source.get("name"),
            split="train",
            streaming=True
        )
        print(f"Yuklendi: {source['label']}")
        return ds
    except Exception as e:
        print(f"Atlanıyor: {source['label']} | Hata: {type(e).__name__}: {e}")
        return None

def main():
    stats = {"tokens": 0, "docs": 0}

    # Sadece açık ve script-hatası üretme olasılığı düşük kaynaklar
    sources = [
        {"label": "Wikipedia TR", "path": "wikimedia/wikipedia", "name": "20231101.tr", "field": "text"},
        {"label": "CC100 TR", "path": "cc100", "name": "tr", "field": "text"},
        {"label": "C4 TR", "path": "allenai/c4", "name": "tr", "field": "text"},
    ]

    try:
        with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
            for source in sources:
                dataset = try_load_stream(source)
                if dataset is None:
                    continue

                print(f"Isleniyor: {source['label']}")
                reached_target = process(dataset, source["field"], f, stats)
                if reached_target:
                    break
    except Exception as e:
        # Dosya sistemi veya beklenmeyen hatalarda bile notebook patlamasın
        print(f"Pipeline guvenli cikis: {type(e).__name__}: {e}")

    print("\nDataset pipeline tamamlandi")
    print(f"docs: {stats['docs']:,}")
    print(f"tokens(est): {stats['tokens']:,}")
    print(f"Saved to: {OUTPUT_FILE}")

if __name__ == "__main__":
    main()